# 02 — Deploy pipelines

Wealth ML SageMaker Foundation (WSF).

Use the **output of notebook 01**, not its metadata CSV.

| Mode | Data input here | What the pipeline's GetData step does |
|---|---|---|
| csv_direct | Generated sample.csv (or another ready flat CSV) | Reads the mounted CSV and copies it unchanged to raw/data.csv |
| feature_store | Feature Store configuration URI saved by notebook 01 | Queries the existing store and writes raw/data.csv |

The feature mapping is used for Feature Store ingestion/query configuration;
CSV-direct does not require it. Both routes hand a flat CSV to the next step: FeatureSelection for training, Preprocess for inference.
The modes share an output contract, not necessarily identical column sets:
Feature Store queries configured model features, or all available predictors when
no features list is supplied. CSV-direct preserves all supplied columns.

Both notebooks read **notebooks/config.json**. Notebook 01 saves the generated data input there.
Run All here after notebook 01; no copy/paste between notebooks is needed. Optional settings are in the **Defaults** cell below.
For an existing CSV or Feature Store binding, set the corresponding config value and run this notebook directly.
This notebook publishes **one shared training pipeline and one shared inference pipeline**.
Adding or renaming a model folder updates their ModelId dropdown when republished.
An existing namespace in config.json is always reused. When it is empty, derive a
stable namespace from your Studio user profile, or from user_alias in config.json.
The names are **<namespace>-training** and **<namespace>-inference**, with outputs
under that namespace's S3 prefix. Re-running the notebook updates those pipelines.
The same user identity produces the same name even with a new config. If the
notebook cannot identify your Studio profile, set user_alias once; it never uses
an anonymous random name or the shared container login. Keep a separate config
for each person. To switch an existing deployment to the user-based name, clear
namespace once; this creates a different pair and leaves the old pipelines intact.
It does not create one pipeline pair per model.

Use a fresh kernel for this checkout; an older checkout can otherwise remain imported.

Model Registry is required. The platform must create a group named
**`<prefix><ModelId>-sm-<environment>`** for each selected model before publication. Set registry_group_prefix in config.json, including its trailing separator.
An empty prefix preserves the original group names.
Training registers **Development / PendingApproval** with separate model approval
**PendingManualApproval**. Each version records the deployment namespace,
environment and ModelId inside the existing shared group.
Approve both fields using the optional cell below or an authorized Studio role.
Inference selects only approved Development models for this namespace and ModelId.
Keep **ModelPackageArn=latest-approved** to select the newest eligible version,
or supply an exact version ARN to replay it. Other namespaces, lifecycle stages and older
versions without this scope metadata are excluded. Train a new version first.

Notebook deployments only create Development versions. The future trusted CDK
path will create ProductionCandidate versions; a notebook flag cannot grant that
authority. BMO Platform must implement the registration/transition permissions.
Keep `include_hpo=False` for now; HPO steps and tuning inputs remain hidden.
Inference defaults to **Processing with the registered model's saved code and image**.
All prediction steps and DriftComparison follow that source release.
Republishing the notebook does not replace an old model's inference code.

Inference runs **ResolveModel → GetData → Preprocess → Score → Postprocess**.
GetData copies the scoring CSV. Preprocess applies the saved feature list and keeps
the primary key; no target is required. Postprocess receives the same raw snapshot,
the saved selection and the score outputs for model-specific analysis.
When training moves to Feature Store, set `data_source="feature_store"` and keep
`inference_data_source="csv_direct"`. Keep both input locations in config.json.

Automatic completion is optional during platform setup. Leave **completion_code_uri**
empty to publish only the pipelines. Training, inference, Model Registry registration
and per-job S3 outputs still run. With tracking_uri configured, jobs log directly to
MLflow; without the completion service, terminal run status and cached-output/final
report consolidation are not updated automatically. Inference still writes its
initial run.json during Postprocess. No follow-up notebook is required to run jobs.

To enable automatic completion, Platform supplies an immutable Lambda ZIP URI built
from operations/completion/build.py inside the approved image. This notebook then
deploys the completion service, requiring CloudFormation/IAM/Lambda/EventBridge
permissions. Platform may instead manage that service through CDK. An empty URI
skips deployment; it does not stop or remove an existing service.
Set **tracking_uri** in config.json to the existing MLflow Tracking Server or App ARN.
This is the only additional tracking input: the namespace supplies experiment names,
and the service supplies artifact locations. Leave it empty to disable tracking.
Before publication, this notebook checks that the Tracking Server is active (when
selected), then uses the installed client to create a **connection-check** run in each
model experiment. It verifies parameters, metrics, dataset references, run search and
artifact upload/download. These labelled runs remain as evidence. Versions are recorded
for diagnostics; equal client/server versions are not required. Notebook success does
not establish job-image compatibility, job-role or human UI access.


## Read your shared inputs

- Read bucket, pipeline role, image and registry_group_prefix from notebooks/config.json.
- Read csv_input or feature_store_config_uri saved there by notebook 01; nothing resets those paths.
- If the file is missing, create a blank template and stop for you to supply values.
- Keep optional settings in the Defaults cell below; target/key/task settings remain in parameters.json.


In [ ]:
from pathlib import Path
import json

published = {}
verified_source_uri = ""

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "mlops-models/src").is_dir())
CONFIG_PATH = ROOT / "notebooks/config.json"
TEMPLATE_PATH = ROOT / "notebooks/config.example.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH.write_text(TEMPLATE_PATH.read_text())
    raise ValueError(f"Created {CONFIG_PATH}. Fill in your inputs in that file, then Run All again. No notebook cell edits are needed.")

# Read supplied inputs; optional settings are declared in the Defaults cell below.
notebook_config = json.loads(TEMPLATE_PATH.read_text()) | json.loads(CONFIG_PATH.read_text())
print("Notebook configuration:", CONFIG_PATH)
bucket = notebook_config["bucket"]
role_arn = notebook_config["role_arn"]
image_uri = notebook_config["image_uri"]
csv_input = notebook_config["csv_input"]
feature_store_config_uri = notebook_config["feature_store_config_uri"]
registry_group_prefix = notebook_config["registry_group_prefix"]
tracking_uri = notebook_config["tracking_uri"].strip()

completion_code_uri = notebook_config.get("completion_code_uri", "").strip()


## Defaults — isolated deployment

- Reuse namespace from config.json, or derive a stable name from your Studio profile or user_alias.
- Save that name and print the S3 folder: bucket/s3_prefix/namespace; s3_prefix defaults to mlops.
- Keep the Registry environment and data sources separate from the deployment name.
- Use uploaded code for Development inference; choose artifact mode to replay the saved model code.


In [ ]:
import hashlib
import re

namespace = notebook_config.get("namespace", "").strip()
if not namespace:
    # A supplied alias also works outside Studio. Do not infer a person from
    # the shared container login or an assumed-role session name.
    deployment_user = notebook_config.get("user_alias", "").strip()
    identity = deployment_user
    if not deployment_user:
        metadata_path = Path("/opt/ml/metadata/resource-metadata.json")
        metadata = json.loads(metadata_path.read_text()) if metadata_path.is_file() else {}
        deployment_user = metadata.get("UserProfileName", "").strip()
        identity = metadata.get("DomainId", "") + "/" + deployment_user
    if not deployment_user:
        raise ValueError(
            "No Studio user profile was found. Set user_alias once in config.json "
            "(for example your username), or supply namespace explicitly."
        )
    label = re.sub(r"[^a-z0-9]+", "-", deployment_user.lower()).strip("-")[:7].rstrip("-") or "user"
    namespace = "wsf-" + label + "-" + hashlib.sha256(identity.encode()).hexdigest()[:8]
    notebook_config["namespace"] = namespace
    CONFIG_PATH.write_text(json.dumps(notebook_config, indent=2) + "\n")
    print("Deployment user:", deployment_user)
print("Reusing deployment namespace:", namespace)

s3_prefix = notebook_config.get("s3_prefix", "mlops").strip().strip("/")
if (not re.fullmatch(r"[A-Za-z0-9_./-]+", s3_prefix)
        or any(part in {"", ".", ".."} for part in s3_prefix.split("/"))):
    raise ValueError("s3_prefix must be a nonempty folder path without '.' or '..' components.")
storage_prefix = f"{s3_prefix}/{namespace}"
print("S3 folder:", storage_prefix)
data_source = "csv_direct"  # Training: change to feature_store when available.
inference_data_source = "csv_direct"  # Scoring snapshots remain explicit CSV inputs.
environment = "dev"
deploy_name = "my-review"  # Reusable label; exact source bytes determine the immutable S3 revision.
model_ids = None
include_hpo = False
tracking_experiment_prefix = namespace
# Adds HasReference → DriftComparison. Requires Evidently in the approved image.
monitoring = {"enabled": True, "prediction_file": "messaging_final.csv", "score_column": "score",
              "threshold": 0.2, "drift_share": 0.5}
cache_steps = True
model_code_source = "artifact"  # Default for the execution dropdown; choose pipeline when testing edits.
universe_data_root = ""
job_role_arn = role_arn  # Set a different job role here if required.
batch_runtime = "processing"  # Native Transform remains optional and needs separate image acceptance.
infrastructure = {
    "network": {"isolation": False, "subnet_ids": [], "security_group_ids": []},
    "output_kms_key_arn": "",
    "volume_kms_key_arn": "",
}
csv_preflight_ok = csv_upload_verified = False
print("Training input:", data_source, "| inference input:", inference_data_source, "| cache_steps:", cache_steps)
print("Inference runtime:", batch_runtime, "| default ModelCodeSource:", model_code_source)

completion_permissions_boundary_arn = ""  # Platform IAM boundary, if required.


## Validate inputs and source

- Show target_key, primary_key and task_type from every selected use case's parameters.json.
- Verify that the requested CSV exists, has unique headers and contains rows; preview it and record its checksum.
- Report target presence without requiring one CSV to contain every use case's target; choose ModelId and its data at execution.
- Check Python syntax and show the checkout, model folders and output root before publishing.


In [ ]:
from uuid import uuid4
import csv
import hashlib
import io
import sys
import boto3
import pandas as pd

csv_preflight_ok = csv_upload_verified = False
verified_source_uri = ""
published = {}
bucket = bucket.strip().removeprefix("s3://").rstrip("/")
role_arn, image_uri = role_arn.strip(), image_uri.strip()
csv_input = csv_input.strip()
print("Effective csv_input:", repr(csv_input))
print("Current notebook directory:", Path.cwd())
print("cache_steps:", cache_steps)
assert bucket and "/" not in bucket and role_arn and image_uri, f"STOP: fill in bucket, role_arn and image_uri in {CONFIG_PATH}."
data_sources = {"training": data_source, "inference": inference_data_source}
assert all(value in {"feature_store", "csv_direct"} for value in data_sources.values()), "STOP: choose csv_direct or feature_store for each pipeline."
uses_csv = "csv_direct" in data_sources.values()
if "feature_store" in data_sources.values():
    assert feature_store_config_uri, f"STOP: supply feature_store_config_uri in {CONFIG_PATH}, or complete notebook 01 in feature_store mode."
if uses_csv:
    assert csv_input, f"STOP: csv_input is empty in {CONFIG_PATH}. Run notebook 01, or set an existing CSV path in that file. Nothing will be published."

sys.path[:0] = [str(ROOT / "mlops-models"), str(ROOT / "mlops-models/src"), str(ROOT / "mlops-features")]
import mlops_models
import mlops_skeleton
for package in (mlops_models, mlops_skeleton):
    assert Path(package.__file__).resolve().is_relative_to(ROOT.resolve()), (
        f"STOP: this kernel loaded code from {package.__file__}. Restart this notebook's kernel for the new checkout."
    )
from mlops_models.pipelines.source import (
    source_identity, source_location, upload_source, put_immutable, build_definitions, pipeline_fingerprints,
    configure_cache_sources, publish_cache_sources,
)
from mlops_skeleton.names import discover_models, validate_namespace
from mlops_models.pipelines import source as source_module

print("Packaging module:", Path(source_module.__file__).resolve())
print("Packaging source directory:", source_module.PACKAGE.resolve())
assert source_module.PACKAGE.resolve() == (ROOT / "mlops-models/src/mlops_models").resolve(), (
    "STOP: the packaging module points at another checkout. Restart the kernel in the intended project."
)

validate_namespace(namespace)
available_models = [folder.name for folder in discover_models(ROOT / "mlops-models/src/mlops_models/models")]
model_ids = available_models if model_ids is None else model_ids
assert model_ids and all(model in available_models for model in model_ids), f"STOP: select model folders from {available_models}."
def csv_settings():
    # Recheck if the CSV, model list or any use-case parameters changed.
    return csv_input, data_source, inference_data_source, tuple(
        (model, (ROOT / "mlops-models/src/mlops_models/models" / model / "parameters.json").read_text())
        for model in model_ids
    )

model_targets = {}
for model in model_ids:
    parameters_path = ROOT / "mlops-models/src/mlops_models/models" / model / "parameters.json"
    parameters = json.loads(parameters_path.read_text())
    for key in ("target_key", "primary_key"):
        assert isinstance(parameters.get(key), str) and parameters[key].strip(), f"STOP: set {key} in {parameters_path}."
    assert parameters.get("task_type") in {"classification", "regression"}, f"STOP: task_type must be classification or regression in {parameters_path}."
    model_targets[model] = parameters["target_key"]
    print("Model:", model, "| target_key:", repr(parameters["target_key"]),
          "| primary_key:", repr(parameters["primary_key"]), "| task_type:", parameters["task_type"])
    print("Parameters:", parameters_path)

for code_file in (ROOT / "mlops-models/src/mlops_models").rglob("*.py"):
    compile(code_file.read_text(), str(code_file), "exec")
print("Model Python syntax checks passed:", model_ids)
pipeline_role_arn = role_arn
job_role_arn = job_role_arn.strip() or role_arn
session = boto3.Session()
sm = session.client("sagemaker")
s3_root = f"s3://{bucket}/{storage_prefix}"
identity = source_identity("direct-deploy", deploy_name)
source_uri = source_location(s3_root, identity)
csv_input_uri, csv_local_path, csv_check = "", None, None
if uses_csv:
    if csv_input.startswith("s3://"):
        csv_input_uri = csv_input
    else:
        csv_local_path = Path(csv_input).expanduser()
        if not csv_local_path.is_absolute():
            csv_local_path = ROOT / csv_local_path
        csv_local_path = csv_local_path.resolve()
        print("Resolved local CSV:", csv_local_path)
        assert csv_local_path.is_file(), f"STOP: CSV file does not exist: {csv_local_path}. Nothing will be published."
    try:
        if csv_local_path is not None:
            csv_bytes = csv_local_path.read_bytes()
            csv_input_uri = f"{s3_root}/inputs/csv-direct/{deploy_name}/sha256-{hashlib.sha256(csv_bytes).hexdigest()}/data.csv"
        else:
            csv_bucket, csv_key = csv_input_uri.removeprefix("s3://").split("/", 1)
            response = session.client("s3").get_object(Bucket=csv_bucket, Key=csv_key)
            with response["Body"] as stream:
                csv_bytes = stream.read()
    except Exception as error:
        raise RuntimeError(f"STOP: cannot read the requested CSV {csv_local_path or csv_input_uri}. Nothing will be published: {error}") from error
    columns = next(csv.reader(io.StringIO(csv_bytes.decode("utf-8-sig"))), [])
    print("CHECKED CSV:", str(csv_local_path or csv_input_uri))
    print("Bytes:", len(csv_bytes), "| Columns:", len(columns))
    print("First 30 column names:", columns[:30])
    print("Model targets from parameters.json:", model_targets)
    assert columns and all(columns), "STOP: CSV has no valid header."
    assert len(set(columns)) == len(columns), "STOP: CSV contains duplicate column names."
    target_presence = {model: target in columns for model, target in model_targets.items()}
    for model, present in target_presence.items():
        print(model, "| target in this CSV:", present, "|", repr(model_targets[model]))
    if not all(target_presence.values()):
        print("This CSV does not contain every configured target. Publication can proceed.")
        print("When training, choose the intended ModelId and its CsvInputUri; that model's code handles its target.")
    preview = pd.read_csv(io.BytesIO(csv_bytes), encoding="utf-8-sig", nrows=5, keep_default_na=False)
    assert len(preview), "STOP: CSV has a header but no data rows."
    display(preview.iloc[:, :10])
    csv_check = {"source": str(csv_local_path or csv_input_uri), "sha256": hashlib.sha256(csv_bytes).hexdigest(),
                 "bytes": len(csv_bytes), "columns": columns, "model_targets": model_targets, "target_presence": target_presence}
    csv_checked_settings = csv_settings()
    csv_preflight_ok = True
    print("CSV FILE CHECK PASSED | SHA-256:", csv_check["sha256"])
    print("Will use S3 CSV:", csv_input_uri)
    del csv_bytes

print("Checkout:", ROOT)
print("Region:", session.region_name)
print("Pipeline role:", pipeline_role_arn)
print("Job role:", job_role_arn)
print("Models:", model_ids)
print("Source upload destination:", source_uri)
print("Training input:", csv_input_uri if data_source == "csv_direct" else feature_store_config_uri)
print("Inference input:", csv_input_uri if inference_data_source == "csv_direct" else feature_store_config_uri)
print("Pipelines:", namespace + "-training", "|", namespace + "-inference")
print("Run outputs:", s3_root + "/models/<ModelId>/runs/<execution-id>/")
print("Source identity:", identity)

## Pin the image

- Resolve a supplied image tag to its immutable digest.
- If only a repository is supplied, explicitly report the newest tagged image.
- Reuse that digest for the published definitions.

In [ ]:
if "@sha256:" not in image_uri:
    registry, repository = image_uri.split("/", 1)
    repository, _, tag = repository.partition(":")
    ecr = session.client("ecr")
    if tag:
        detail = ecr.describe_images(
            registryId=registry.split(".")[0], repositoryName=repository,
            imageIds=[{"imageTag": tag}],
        )["imageDetails"][0]
    else:
        # Repository address only: use its newest tagged image,.
        images = []
        for page in ecr.get_paginator("describe_images").paginate(
                registryId=registry.split(".")[0], repositoryName=repository,
                filter={"tagStatus": "TAGGED"}):
            images.extend(page["imageDetails"])
        assert images, "That repository has no tagged images; paste a full image URI instead."
        detail = max(images, key=lambda item: item["imagePushedAt"])
        print("Pinned the newest tagged image:", detail.get("imageTags"))
    image_uri = registry + "/" + repository + "@" + detail["imageDigest"]
print("Image:", image_uri)

## Check tracking and build the shared pipelines

- When tracking_uri is set, run a visible connection check using this Python environment and its normal AWS credential chain; stop the worker after 120 seconds if it has not finished.
- Verify runs, metrics, parameters, dataset references and artifacts in each model experiment; retain the small connection-check runs for inspection.
- Build the shared pipeline definitions and verify access to existing Registry groups; the check does not create a tracking server or configure MLflow Model Registry.
- Save definitions locally and display cache settings; published jobs log to MLflow independently, and notebook access does not establish job-role access.


In [ ]:
if uses_csv:
    assert csv_preflight_ok and csv_checked_settings == csv_settings(), "STOP: rerun the CSV check after changing the input, selected models or their targets."

validate_namespace(environment)
tracking = None
tracking_check = None
tracking_artifact_uris = []
if tracking_uri:
    print("Preparing MLflow notebook check...", flush=True)
    from mlops_models.pipelines.tracking import check_tracking_notebook, validate_tracking
    tracking = validate_tracking({"uri": tracking_uri, "experiment_prefix": tracking_experiment_prefix})
    tracking_check = check_tracking_notebook(tracking, [
        f"{tracking_experiment_prefix}/{environment}/{model}" for model in model_ids
    ], timeout_seconds=120)
    tracking["client_version"] = tracking_check["client_version"]
    tracking_artifact_uris = [item["artifact_location"] for item in tracking_check["experiments"]]
    print("MLflow notebook write/read check passed:", json.dumps(tracking_check, indent=2))
    print("Jobs use the MLflow client and sagemaker-mlflow plugin supplied in their image.")
    print("Job client compatibility, role access and connectivity are verified when jobs execute; UI access is separate.")
else:
    print("MLflow is disabled. Set tracking_uri in config.json to enable it.")
registry_groups = {}
for model in model_ids:
    group_name = f"{registry_group_prefix}{model}-sm-{environment}"
    print("Checking existing Registry group:", group_name)
    group = sm.describe_model_package_group(ModelPackageGroupName=group_name)
    registry_groups[model] = group["ModelPackageGroupArn"]
    print("Registry group:", registry_groups[model])
print("Group reads passed for the notebook caller. Pipeline registration and job reads are verified when executed.")

step_cache_revisions = {}
if cache_steps:
    step_cache_revisions = pipeline_fingerprints(session, csv_input_uri, csv_local_path)
print("Compute step cache:", "enabled for 24 hours" if step_cache_revisions else "disabled")
if step_cache_revisions:
    print("Step revisions:", step_cache_revisions)
    print("Training: GetData, FeatureSelection, Preprocess, Train, Evaluate.")
    print("Processing batch inference: GetData, Preprocess, Score. Postprocess runs for every execution.")
    print("Registry checks, registration and conditions run normally. Feature Store reads remain uncached.")
    print("Unchanged compute steps retain their source snapshot across publications.")

definitions = build_definitions(
    namespace, job_role_arn, image_uri, s3_root, source_uri, identity,
    region=session.region_name, models=model_ids, infrastructure=infrastructure, batch_runtime=batch_runtime,
    feature_store_config_uri=feature_store_config_uri, universe_data_root=universe_data_root,
    data_source=data_source, inference_data_source=inference_data_source, csv_input_uri=csv_input_uri,
    step_cache_revisions=step_cache_revisions,
    model_code_source=model_code_source,
    environment=environment, registry_group_prefix=registry_group_prefix, include_hpo=include_hpo,
    tracking=tracking, monitoring=monitoring,
)
cache_source_bindings = configure_cache_sources(definitions)
published_cache_sources = []
WORK = ROOT / ".runs/source-deploy" / namespace
WORK.mkdir(parents=True, exist_ok=True)
for kind, body in definitions.items():
    path = WORK / (kind + ".json")
    path.write_text(json.dumps(body, indent=2) + "\n")
    print(namespace + "-" + kind, "→", " → ".join(step["Name"] for step in body["Steps"]))
    print("Definition:", path.resolve())
print("Model Registry: Development versions only; inference requires namespace, stage and model approval.")
print("Registry namespace:", namespace)
print("Registry groups:", list(registry_groups.values()))

if monitoring.get("enabled"):
    print("After Postprocess: ReferenceExecutionArn=none skips DriftComparison; supply a prior execution ARN to compare.")
print("Batch inference runtime:", batch_runtime)
print("Default ModelCodeSource:", model_code_source)
print("Development Processing executions can choose pipeline to test published edits without training.")
if batch_runtime == "both":
    print("InferenceRuntime=processing: ResolveModel → GetData → Preprocess → Score → Postprocess")
    print("InferenceRuntime=transform: ResolveModel → GetDataTransform → PreprocessTransform → CreateModel → Transform → PostprocessTransform")
    print("Start two executions with the same ModelId and CsvInputUri to test the routes concurrently.")
if batch_runtime in {"both", "transform"}:
    print("Transform uses archived code. Train/package and approve a version containing the updated serve adapter.")
    print("The image must route serve to the model server and include Flask/Gunicorn and model dependencies.")
    print("TransformOutputFile defaults to messaging_final.csv; use a relative path if needed.")
if cache_steps and not infrastructure.get("network", {}).get("isolation", False):
    print("Cached Training needs sagemaker:DescribeTrainingJob on the job execution role.")
print("HPO branch:", "included; UseHPO defaults to false" if include_hpo else "hidden; ordinary training only")

## Upload verified input and source

- Upload the selected local CSV, or verify the supplied S3 CSV.
- Compare the uploaded CSV bytes with the preflight checksum.
- Upload runtime source and read back its completion receipt.
- Publish immutable source bindings for cached steps; print which steps retain earlier source snapshots.


In [ ]:
csv_upload_verified = False
verified_source_uri = ""
published = {}
published_cache_sources = []
if uses_csv:
    assert csv_preflight_ok and csv_checked_settings == csv_settings(), "STOP: rerun the CSV check after changing the input, selected models or their targets."
    csv_bucket, csv_key = csv_input_uri.removeprefix("s3://").split("/", 1)
    try:
        if csv_local_path is not None:
            csv_upload_bytes = csv_local_path.read_bytes()
            assert hashlib.sha256(csv_upload_bytes).hexdigest() == csv_check["sha256"], "STOP: local CSV changed after validation; rerun the notebook."
            kms = infrastructure.get("output_kms_key_arn", "")
            encryption = {"ServerSideEncryption": "aws:kms", "SSEKMSKeyId": kms} if kms else {}
            put_immutable(session.client("s3"), csv_bucket, csv_key, csv_upload_bytes, **encryption)
            del csv_upload_bytes
        response = session.client("s3").get_object(Bucket=csv_bucket, Key=csv_key)
        with response["Body"] as stream:
            uploaded_sha256 = hashlib.sha256(stream.read()).hexdigest()
        assert uploaded_sha256 == csv_check["sha256"], "S3 CSV differs from the file checked above."
    except Exception as error:
        raise RuntimeError(f"STOP: CSV upload/readback verification failed at {csv_input_uri}. Pipelines will not be published: {error}") from error
    csv_upload_verified = True
    print("S3 CSV VERIFIED:", csv_input_uri)
    print("SHA-256:", uploaded_sha256, "| Columns:", len(csv_check["columns"]))
uploaded_source = upload_source(session, s3_root, identity, expected_uri=source_uri)
source_bucket, source_prefix = uploaded_source.removeprefix("s3://").split("/", 1)
with session.client("s3").get_object(Bucket=source_bucket, Key=source_prefix.rstrip("/") + "/source.json")["Body"] as stream:
    source_receipt = json.load(stream)
assert source_receipt["uri"] == source_uri, "STOP: source upload receipt names a different location."
assert source_receipt["sha256"] == source_uri.rsplit("/sha256-", 1)[1], "STOP: source upload receipt has a different hash."
assert all(source_receipt.get(key) == value for key, value in identity.items()), "STOP: source upload receipt has a different identity."
published_cache_sources = publish_cache_sources(session, s3_root, uploaded_source, cache_source_bindings)
for binding in published_cache_sources:
    print("Step source:", binding["kind"], binding["step"], "→", binding["source_uri"],
          "(retained)" if binding["reused_source"] else "(current publication)")
verified_source_uri = uploaded_source
print("SOURCE UPLOAD VERIFIED:", verified_source_uri)
print("Source SHA-256:", source_receipt["sha256"])
print("GetData sources:", data_sources)

## Publish in Studio

- When completion_code_uri is supplied, deploy the completion Lambda and its event/reconciliation rules; otherwise skip service deployment.
- Create or update the same namespace-training and namespace-inference pipelines.
- Require a completed source upload with unchanged local source, then verify saved definitions and roles; no execution is started.
- Save the publication record and print the ModelId choices.

In [ ]:
published = {}
assert verified_source_uri == source_uri, "STOP: complete the source upload cell successfully before publishing."
assert source_location(s3_root, identity) == verified_source_uri, "STOP: local source changed after upload; rerun the notebook."
assert {(b["kind"], b["step"], b["revision"]) for b in published_cache_sources} == {
    (b["kind"], b["step"], b["revision"]) for b in cache_source_bindings
}, "STOP: complete source upload and cache bindings before publishing."
for kind, body in definitions.items():
    saved_source = next((p.get("DefaultValue") for p in body["Parameters"]
                         if p["Name"] == "PipelineSourceUri"), None)
    assert saved_source == verified_source_uri, (
        f"STOP: {kind} PipelineSourceUri differs from the uploaded source. "
        "Restart the kernel and run all cells with the current pipelines/source.py."
    )
if uses_csv:
    assert csv_upload_verified and csv_checked_settings == csv_settings(), "STOP: complete CSV verification before publishing."
    for body in definitions.values():
        if body["Metadata"]["DataSource"] != "csv_direct":
            continue
        saved_csv = next(p["DefaultValue"] for p in body["Parameters"] if p["Name"] == "CsvInputUri")
        assert saved_csv == csv_input_uri, "STOP: definition uses another CSV; rerun all cells."
completion = {"deployment": "skipped"}
if completion_code_uri:
    sys.path.insert(0, str(ROOT))
    from operations.completion.deploy import deploy_completion
    completion = deploy_completion(
        session, namespace, s3_root, completion_code_uri, tracking_uri=tracking_uri,
        tracking_artifact_uris=tracking_artifact_uris,
        registry_prefix=registry_group_prefix, kms_key=infrastructure.get("output_kms_key_arn", ""),
        permissions_boundary_arn=completion_permissions_boundary_arn,
    )
else:
    print("Completion service deployment skipped. Existing services are unchanged.")
    print("Jobs and Model Registry work independently. Automatic final S3/MLflow status requires the Platform service.")
published = {}
for kind, body in definitions.items():
    name = f"{namespace}-{kind}"
    request = {
        "PipelineName": name, "RoleArn": pipeline_role_arn,
        "PipelineDefinition": json.dumps(body),
    }
    try:
        existing = sm.describe_pipeline(PipelineName=name)
    except sm.exceptions.ClientError as error:
        if "does not exist" not in error.response["Error"]["Message"]:
            raise
        response = sm.create_pipeline(**request, Tags=[
            {"Key": "Deployment", "Value": namespace},
            {"Key": "SourceDeploy", "Value": identity["mode"]},
        ])
        version = 1  # The initial version of a newly created pipeline.
    else:
        tags = {t["Key"]: t["Value"] for t in sm.list_tags(ResourceArn=existing["PipelineArn"])["Tags"]}
        if tags.get("Deployment") != namespace or tags.get("SourceDeploy") != identity["mode"]:
            raise ValueError(f"{name} belongs to another deployment.")
        response = sm.update_pipeline(**request)
        version = response["PipelineVersionId"]
    actual = sm.describe_pipeline(PipelineName=name, PipelineVersionId=version)
    assert json.loads(actual["PipelineDefinition"]) == body, "Published definition differs; inspect before executing."
    assert actual["RoleArn"] == pipeline_role_arn, "Saved pipeline role differs; inspect before executing."
    published[kind] = {"name": name, "arn": response["PipelineArn"], "version": version}
    print("Published:", name, "| version:", version, "| saved role:", actual["RoleArn"])

publication = {
    "namespace": namespace, "registry_stage": "Development", "pipelines": published, "models": model_ids,
    "tracking": tracking, "tracking_check": tracking_check, "completion": completion, "monitoring": monitoring, "job_role_arn": job_role_arn, "infrastructure": infrastructure,
    "source": identity, "source_uri": source_uri, "image": image_uri,
    "s3_root": s3_root, "batch_runtime": batch_runtime, "region": session.region_name,
    "environment": environment, "registry_group_prefix": registry_group_prefix, "registry_groups": registry_groups, "include_hpo": include_hpo,
    "feature_store_config_uri": feature_store_config_uri, "universe_data_root": universe_data_root,
    "data_source": data_source, "inference_data_source": inference_data_source, "csv_input_uri": csv_input_uri,
    "step_cache_revisions": step_cache_revisions, "cache_sources": published_cache_sources,
    "model_code_source": definitions["inference"]["Metadata"]["ModelCodeSource"],
    "csv_check": csv_check,
}
path = WORK / "publication.json"
path.write_text(json.dumps(publication, indent=2) + "\n")
print("Publication complete:", path.resolve())
print("One training pipeline and one inference pipeline are ready in Studio.")
print("ModelId dropdown:", model_ids)
print("Train here, then approve both Development stage status and model approval for the exact version.")
print("For an already approved model, start Processing inference with its ModelId and scoring CsvInputUri.")
if batch_runtime == "both":
    print("InferenceRuntime dropdown: processing (default), transform. Each execution runs only its chosen branch.")
print("PipelineSourceUri in both pipelines:", source_uri)
print("Compare that parameter with Uploaded source above; CsvInputUri is a separate data input.")
if model_code_source == "artifact":
    print("ModelCodeSource defaults to artifact: saved model code. Select pipeline on a Development execution to test edits.")
    print("To test published Processing edits: select ModelCodeSource=pipeline when starting inference; no training needed.")
else:
    print("Processing uses the source bindings published for this pipeline version.")
    print("Unchanged cached steps can retain an older source snapshot; see Step source above.")
    print("Native Transform uses the selected model's saved code for its entire route.")
print("Inference resolves the newest approved Development version in this namespace, or the exact ModelPackageArn.")
if uses_csv:
    print("Use the newly printed pipeline version and this CsvInputUri:", csv_input_uri)
    print("A previous execution/version or a Studio parameter override can select a different CSV.")

## Execute and review Development models

1. Open the printed training pipeline in SageMaker Studio and execute the published
   version with ModelId and its CsvInputUri. Training must pass evaluation and registration.
2. In **Models → Registered models**, open the existing group and the new version.
   Check its Development stage and DeploymentNamespace/ModelId metadata.
3. Review its outputs. Set **ModelApprovalStatus=Approved**, then use **Update model
   lifecycle** to keep **Stage=Development** and set **StageStatus=Approved**.
   The optional cell below performs those two updates together for an exact version.
4. Start this namespace's inference pipeline with ModelId and the scoring CSV.
   Keep **ModelPackageArn=latest-approved**, or supply an exact version ARN to replay it. Selection never falls back to another namespace/stage.

These checks define this demo's model selection behavior. They do not replace IAM:
the Platform team must restrict stage changes, source writes and registration-role
use before treating ProductionCandidate as a protected production boundary.

Development Processing inference exposes **ModelCodeSource** on each execution.
The notebook default is **artifact**, which imports the registered model's saved code.
Select **pipeline** to run published inference edits without training or registering a new model.
**PipelineSourceUri** identifies this publication. Cached GetData, Preprocess and Score
resolve their own source bindings; unchanged code can retain an older snapshot.
Postprocess and DriftComparison use the current publication.
After editing code, republish and start a new execution. Registry still supplies
approved model metadata, fitted objects and the saved image; those are not replaced.
You are responsible for compatibility between edited code and those fitted objects.

Select **ModelCodeSource=artifact** for saved-code replay. The development loader
still comes from the publication, so it can apply the choice and verify metadata.
CI deployments require artifact mode and pin the deployment-source parameter.
This notebook still registers/selects Development models only. ProductionCandidate
promotion and its trusted CI/IAM route remain Platform integration work.
Keep the saved source, model artifact and ECR image for historical replay.

Processing runs **ResolveModel → GetData → Preprocess → Score → Postprocess**.
The resolver executes every time; matching compute can be cached. Postprocess
runs customer analysis and execution reporting. A HasReference gate then runs
DriftComparison only when ReferenceExecutionArn differs from none. The comparison uses
the selected code mode and saved model image. Both jobs write to S3 and the same inference
MLflow run when tracking is enabled.

Keep **ReferenceExecutionArn=none** to skip DriftComparison entirely. For comparison,
supply a prior successful inference ARN from this namespace. Keep these nonempty defaults
in Studio: explicitly submitting a blank parameter value causes an execution-start validation error.
The job reads retained reference outputs,
including cached producers; it does not score the reference again.

Native Transform remains optional. Its image must support the serving entrypoint;
if batch_runtime="both", select InferenceRuntime for each execution and use the
same explicit ModelPackageArn when comparing routes. HPO similarly requires its
approved container startup contract and separate acceptance.

When provisioned, EventBridge and the completion Lambda finalize MLflow and run.json
automatically; reconciliation retries missed or unsuccessful completions every five
minutes. Without that service, inspect native pipeline status and per-job outputs.
MLflow can remain RUNNING after the native execution ends, and an entirely cached
training run may have no new MLflow run. No notebook needs to remain open or be run
after an execution for the model jobs and Model Registry registration to complete.
Denied Registry calls, missing scope/approval or missing retained source fail
visibly. Versions from before Development/namespace metadata was introduced remain
untouched and are not selected by these new pipelines.


## Optional exact-version review and approval

- Paste the version ARN produced by training; leave it empty to make no request.
- Check its group, namespace, ModelId, environment and Development stage.
- Set approve_for_development=True only after reviewing that version's outputs.
- Approval updates both approval fields and keeps the model in Development.

**Effect:** the enabled action updates only the named Development model using your
current role. It never promotes a model to ProductionCandidate.


In [ ]:
model_package_arn = ""  # Exact version ARN from this namespace's training.
approve_for_development = False
if model_package_arn:
    from mlops_models.registry.model_package import development_scope
    package = sm.describe_model_package(ModelPackageName=model_package_arn)
    selected_model = package.get("CustomerMetadataProperties", {}).get("ModelId", "")
    group_name = f"{registry_group_prefix}{selected_model}-sm-{environment}"
    if selected_model not in model_ids or not development_scope(
            package, group_name, namespace, selected_model, environment):
        raise ValueError("This is not a completed Development model from this notebook's namespace.")
    print("Version:", model_package_arn)
    print("Lifecycle:", package["ModelLifeCycle"], "| model approval:", package["ModelApprovalStatus"])
    print("Scope and source:", package["CustomerMetadataProperties"])
    if approve_for_development:
        sm.update_model_package(
            ModelPackageArn=model_package_arn,
            ModelApprovalStatus="Approved",
            ModelLifeCycle={"Stage": "Development", "StageStatus": "Approved"})
        print("Approved for Development inference:", model_package_arn)
else:
    print("No model selected or approval changed.")


## Confirm publication

- Read the pipeline names and last update times directly from AWS, with timestamps in UTC.
- Compare the current AWS definitions with this notebook's deployment and show its published versions.
- Compare the saved PipelineSourceUri parameter with the exact uploaded source URI; CsvInputUri is data.
- Show which steps allow caching and whether inference loads uploaded code or the saved model archive.

**Effect:** reads deployed settings; no pipeline execution is started.


In [ ]:
from datetime import datetime, timezone

from mlops_models.pipelines.source import walk_steps

print("PIPELINE PUBLICATION CHECK")
print("Checked at:", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"))
print("Notebook source directory:", ROOT / "mlops-models/src")
print("Uploaded source:", source_uri)
source_matches_local = verified_source_uri == source_uri and source_location(s3_root, identity) == source_uri
print("Verified upload matches current local source:", "YES" if source_matches_local else "NO — rerun upload and publication")
publication_complete = bool(definitions) and set(published) == set(definitions)
print("All pipelines published in this run:", "YES" if publication_complete else "NO — publication did not finish")
all_definitions_match = publication_complete and source_matches_local
for kind, deployed in published.items():
    saved = sm.describe_pipeline(PipelineName=deployed["name"])
    saved_definition = json.loads(saved["PipelineDefinition"])
    matches = saved_definition == definitions[kind]
    saved_source = next((p.get("DefaultValue") for p in saved_definition["Parameters"]
                         if p["Name"] == "PipelineSourceUri"), None)
    source_parameter_matches = saved_source == verified_source_uri
    all_definitions_match = all_definitions_match and matches and source_parameter_matches
    cached_steps = [
        step["Name"] for step in walk_steps(saved_definition["Steps"])
        if step.get("CacheConfig", {}).get("Enabled", False)
    ]
    print()
    print("Pipeline:", saved["PipelineName"])
    print("ARN:", saved["PipelineArn"])
    print("Version published by this notebook:", deployed["version"])
    print("Last updated in AWS:",
          saved["LastModifiedTime"].astimezone(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"))
    print("Current AWS definition matches this notebook:", "YES" if matches else "NO — rerun publication or inspect the newer deployment")
    print("PipelineSourceUri:", saved_source)
    print("PipelineSourceUri matches Uploaded source:", "YES" if source_parameter_matches else "NO — rerun publication")
    print("Caching enabled for:", ", ".join(cached_steps) if cached_steps else "none — steps run again")
    print("Unchanged cached steps may use a retained source snapshot from an earlier publication.")
    if kind == "inference":
        saved_code_source = saved_definition.get("Metadata", {}).get("ModelCodeSource")
        print("Default ModelCodeSource:", saved_code_source)
        if saved_code_source == "artifact":
            print("Uses code saved in the approved model.tar.gz. Local print/code edits are not used.")
            print("For a Development Processing execution, set ModelCodeSource=pipeline to use published edits without retraining.")
        elif saved_code_source == "pipeline":
            print("Processing uses the published source binding for each step with the registered model's fitted artifacts.")
            print("This includes GetData. The optional native Transform route still uses saved model code.")

print()
print("PIPELINE PUBLICATION CONFIRMED" if all_definitions_match else "PUBLICATION CHECK FOUND A DIFFERENCE")
if all_definitions_match:
    print("Start a new execution using the pipeline names and published versions above.")
    print("In Studio's execution parameters, PipelineSourceUri must match Uploaded source above.")
    print("CsvInputUri points to input data and can have a different hash.")
print("To verify execution code, use mlops-models/scratch/trace_execution_source.py with its ARN and the file you edited.")


## Inspect the source selected by a Processing job

- Paste the Processing job name linked from the execution's Preprocess step into JOB_NAME below.
- Read that job's code mode, entrypoint, status and creation time.
- Print the actual S3 input paths and their locations inside the container.
- For a cache hit, inspect the original producing job; no new job ran for the reused step.

**Effect:** reads job metadata only. With JOB_NAME empty, this cell does nothing beyond printing instructions.


In [ ]:
JOB_NAME = ""  # Paste the Processing job name from the Preprocess step.

if not JOB_NAME.strip():
    print("Enter the Preprocess Processing job name in JOB_NAME, then rerun this cell.")
else:
    import boto3

    job = boto3.client("sagemaker").describe_processing_job(
        ProcessingJobName=JOB_NAME.strip()
    )
    print("PROCESSING JOB SOURCE CHECK")
    print("Job:", job["ProcessingJobName"])
    print("Status:", job["ProcessingJobStatus"])
    print("Created:", job["CreationTime"])
    print("Code mode:", job.get("Environment", {}).get("MLOPS_MODEL_CODE_SOURCE", "not set"))
    print("Entrypoint:", job["AppSpecification"].get("ContainerEntrypoint"))

    for item in job.get("ProcessingInputs", []):
        if "S3Input" in item:
            source = item["S3Input"]
            print(item["InputName"], ":", source["S3Uri"], "→", source["LocalPath"])
